# 🚀 Entrenamiento Interactivo de U-Net

Este notebook permite entrenar U-Net de forma interactiva usando la clase `UNetTrainer`.

**Ventajas vs script CLI:**
- Experimentación interactiva
- Visualización en tiempo real
- Debugging más fácil
- Control total sobre hiperparámetros

## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ⚡ OPTIMIZACIÓN CRÍTICA: Activar cuDNN benchmark
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from scripts.utils import (
    get_device, set_seed, load_class_weights,
    create_dataloaders, get_training_augmentation, get_validation_augmentation, get_light_augmentation,
    UNetTrainer, N_CLASSES,
    PATCHES_DIR, PATCHES_INDEX
)

# Configurar
set_seed(42)
device = get_device()

print(f"Device: {device}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_vram = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"VRAM Total: {total_vram:.2f} GB")
    print(f"VRAM Libre: {free_vram:.2f} GB")
    
    # Recomendación de batch size para LAPTOP GPUs
    # IMPORTANTE: Laptop GPUs requieren batch sizes más conservadores que desktop GPUs
    if total_vram >= 12:
        rec_batch = "16-24"
    elif total_vram >= 8:
        rec_batch = "8 (ÓPTIMO para RTX 4060 Laptop)"
    elif total_vram >= 6:
        rec_batch = "4-6"
    else:
        rec_batch = "2-4"
    print(f"Batch size recomendado: {rec_batch}")
    print(f"\nNOTA: Para RTX 4060 Laptop con 8GB VRAM, usar BATCH_SIZE=8")

## 2. Cargar Class Weights

In [ ]:
class_weights = load_class_weights()

print(f"Class weights:")
print(class_weights.numpy())

## 3. Crear DataLoaders

In [ ]:
# ⚡ CONFIGURACIÓN ÓPTIMA para RTX 4060 Laptop (8GB VRAM)
# ============================================================
# CRÍTICO: batch_size=8 es ÓPTIMO para esta GPU

# Hiperparámetros del DataLoader
BATCH_SIZE = 8  # ⚡ ÓPTIMO para RTX 4060 Laptop
NUM_WORKERS = 0  # ⚠️ CRÍTICO en Windows: DEBE SER 0

# Data augmentation
train_transform = get_training_augmentation()  # Augmentaciones completas
val_transform = get_validation_augmentation()

print("⚡ CONFIGURACIÓN PARA RTX 4060 LAPTOP:")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Num workers: {NUM_WORKERS} (0 para Windows)")
print(f"  - Augmentation: COMPLETA")
print()

# Crear dataloaders
dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_transform=train_transform,
    val_transform=val_transform,
    use_balanced_sampler=False  # Cambiar a True para sobre-muestrear minoritarias
)

train_loader = dataloaders['train']
val_loader = dataloaders['val']
test_loader = dataloaders['test']

print(f"✓ DataLoaders creados:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")
print()
print(f"Tiempo estimado por epoch: {len(train_loader) * 0.3 / 60:.1f} - {len(train_loader) * 0.5 / 60:.1f} minutos")

## 4. Crear Modelo U-Net

In [ ]:
# Configuración del modelo
ENCODER = 'resnet34'  # Opciones: resnet18, resnet34, resnet50, efficientnet-b0, etc.
ENCODER_WEIGHTS = 'imagenet'  # Pre-entrenado en ImageNet

# Crear modelo
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=N_CLASSES
)

# ⚡ MOVER MODELO A GPU
model = model.to(device)
print(f"✓ Modelo movido a: {device}")

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Modelo U-Net creado:")
print(f"  Encoder: {ENCODER}")
print(f"  Pre-entrenado: {ENCODER_WEIGHTS}")
print(f"  Total parámetros: {total_params:,}")
print(f"  Parámetros entrenables: {trainable_params:,}")
print(f"  Device: {next(model.parameters()).device}")

## 4.1. ⚡ Verificar GPU y Velocidad

In [ ]:
# ⚡ VERIFICACIÓN CRÍTICA: GPU y velocidad del DataLoader
import time

print("="*60)
print("VERIFICACIÓN DE GPU Y VELOCIDAD")
print("="*60)

# 1. Verificar que el modelo está en GPU
model_device = next(model.parameters()).device
print(f"\n1. Modelo en device: {model_device}")
if model_device.type != 'cuda':
    print("   WARNING: Modelo NO está en GPU!")
    model = model.to(device)
    print(f"   ✓ Modelo movido a: {device}")
else:
    print("   ✓ Modelo correctamente en GPU")

# 2. Test de velocidad del DataLoader
print(f"\n2. Test de velocidad del DataLoader:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Num workers: {NUM_WORKERS}")
print()

# Medir tiempo de 10 batches
start = time.time()
for i, (images, masks) in enumerate(train_loader):
    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True)
    
    # Forward pass simple
    with torch.no_grad():
        outputs = model(images)
    
    if i >= 9:  # 10 batches (0-9)
        break

elapsed = time.time() - start
avg_time = elapsed / 10

print(f"   Tiempo para 10 batches: {elapsed:.2f}s")
print(f"   Tiempo promedio por batch: {avg_time:.2f}s")
print()

# 3. Estimación de tiempo de entrenamiento
total_batches = len(train_loader)
estimated_time_per_epoch = (total_batches * avg_time) / 60  # en minutos

print(f"3. Estimación de tiempos:")
print(f"   Total batches: {total_batches}")
print(f"   Tiempo estimado por epoch: {estimated_time_per_epoch:.1f} minutos")
print(f"   Tiempo para 100 epochs: {estimated_time_per_epoch * 100 / 60:.1f} horas")
print()

# 4. Evaluación
if avg_time > 1.0:
    print("WARNING: Muy lento! (>1s por batch)")
    print("   Sugerencias:")
    print("   - Verificar que NUM_WORKERS = 0")
    print("   - Verificar que GPU está en modo 'Máximo rendimiento'")
    print("   - Posible problema con instalación de PyTorch/CUDA")
elif avg_time > 0.5:
    print("Velocidad ACEPTABLE (0.5-1s por batch)")
    print(f"   Con batch_size={BATCH_SIZE}, esto es razonable para laptop GPU")
else:
    print("✓ Velocidad EXCELENTE! (<0.5s por batch)")
    print("   El entrenamiento será rápido")

print("="*60)

## 5. Crear Trainer

In [ ]:
# Hiperparámetros de entrenamiento
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EXPERIMENT_NAME = 'unet_resnet34_baseline'
PATIENCE = 15  # Early stopping

# Crear trainer
trainer = UNetTrainer(
    model=model,
    device=device,
    class_weights=class_weights,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    experiment_name=EXPERIMENT_NAME,
    patience=PATIENCE
)

print(f"\n✓ Trainer listo para entrenar!")
print(f"  TensorBoard: {trainer.log_dir}")

In [ ]:
import types

def train_with_resume(self, train_loader, val_loader, num_epochs, start_epoch=0):
    """
    Loop principal de entrenamiento con soporte para reanudar
    
    Args:
        train_loader: DataLoader de entrenamiento
        val_loader: DataLoader de validación
        num_epochs: Número total de épocas (no adicionales)
        start_epoch: Epoch desde el cual retomar (0 = desde el inicio)
    """
    if start_epoch > 0:
        print("\n" + "="*80)
        print(f"RETOMANDO ENTRENAMIENTO DESDE EPOCH {start_epoch + 1}")
        print("="*80)
    else:
        print("\n" + "="*80)
        print("INICIANDO ENTRENAMIENTO")
        print("="*80)
    
    for epoch in range(start_epoch + 1, num_epochs + 1):
        # Entrenar
        train_metrics = self.train_epoch(train_loader, epoch)
        
        # Validar
        val_metrics = self.validate_epoch(val_loader, epoch)
        
        # Actualizar scheduler
        if self.scheduler:
            self.scheduler.step()
        
        # Logging
        current_lr = self.optimizer.param_groups[0]['lr']
        self.history['train_loss'].append(train_metrics['loss'])
        self.history['val_loss'].append(val_metrics['loss'])
        self.history['train_miou'].append(train_metrics['miou'])
        self.history['val_miou'].append(val_metrics['miou'])
        self.history['lr'].append(current_lr)
        
        self.log_metrics(epoch, train_metrics, val_metrics)
        
        # Imprimir resumen
        print(f"\nEpoch {epoch}/{num_epochs}:")
        print(f"  Train - Loss: {train_metrics['loss']:.4f}, mIoU: {train_metrics['miou']:.4f}")
        print(f"  Val   - Loss: {val_metrics['loss']:.4f}, mIoU: {val_metrics['miou']:.4f}")
        print(f"  LR: {current_lr:.2e}")
        
        # Early stopping y checkpoint
        val_miou = val_metrics['miou']
        is_best = val_miou > self.best_val_miou
        
        if is_best:
            self.best_val_miou = val_miou
            self.epochs_without_improvement = 0
            self.save_checkpoint(epoch, is_best=True)
        else:
            self.epochs_without_improvement += 1
        
        # Guardar último checkpoint cada 10 epochs
        if epoch % 10 == 0:
            self.save_checkpoint(epoch, is_best=False)
        
        # Verificar early stopping
        if self.epochs_without_improvement >= self.patience:
            print(f"\n⚠ Early stopping activado después de {epoch} epochs")
            print(f"Mejor mIoU: {self.best_val_miou:.4f}")
            break
    
    # Guardar historial
    self.save_training_history()
    
    # Cerrar TensorBoard
    self.writer.close()
    
    print("\n" + "="*80)
    print("ENTRENAMIENTO COMPLETADO")
    print("="*80)
    print(f"Mejor mIoU: {self.best_val_miou:.4f}")
    
    return self.history

# Parchear el método train del trainer
trainer.train = types.MethodType(train_with_resume, trainer)
print("✓ Método train parcheado para soportar reanudación desde epoch específico")

In [ ]:
# Número de epochs
NUM_EPOCHS = 100  # Se detendrá antes si early stopping se activa

# ENTRENAR (retomará desde start_epoch si se cargó un checkpoint)
# Si no hay checkpoint, start_epoch será 0 y empezará desde el principio
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    start_epoch=start_epoch if 'start_epoch' in dir() else 0
)

print(f"\n✓ Entrenamiento completado!")
print(f"Mejor mIoU: {trainer.best_val_miou:.4f}")

In [ ]:
from pathlib import Path

# Cargar el mejor checkpoint para retomar entrenamiento
checkpoint_path = Path(f'../models/checkpoints/{EXPERIMENT_NAME}/best_checkpoint.pth')

if checkpoint_path.exists():
    start_epoch = trainer.load_checkpoint(checkpoint_path)
    print(f"\n✓ Retomando entrenamiento desde epoch {start_epoch + 1}")
    print(f"Mejor mIoU hasta ahora: {trainer.best_val_miou:.4f}")
    print(f"Historial cargado con {len(trainer.history['train_loss'])} epochs previos")
else:
    start_epoch = 0
    print(f"⚠ No se encontró checkpoint en: {checkpoint_path}")
    print(f"Iniciando entrenamiento desde epoch 1")

## 5.1. 🔄 Cargar Checkpoint (para retomar entrenamiento)

**Ejecuta esta celda SOLO si quieres retomar un entrenamiento previo.**

Si es la primera vez que entrenas, salta a la sección 6.

## 6. Entrenar Modelo

**Nota:** Esto tomará varias horas. Puedes monitorear en TensorBoard:

```bash
tensorboard --logdir=logs/tensorboard
```

In [ ]:
# Número de epochs
NUM_EPOCHS = 100  # Se detendrá antes si early stopping se activa

# ENTRENAR
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS
)

print(f"\n✓ Entrenamiento completado!")
print(f"Mejor mIoU: {trainer.best_val_miou:.4f}")

## 7. Visualizar Historial de Entrenamiento

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

# mIoU
axes[1].plot(history['train_miou'], label='Train')
axes[1].plot(history['val_miou'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mIoU')
axes[1].set_title('Mean IoU')
axes[1].legend()
axes[1].grid(True)

# Learning Rate
axes[2].plot(history['lr'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"\nMejor mIoU en validación: {max(history['val_miou']):.4f}")
print(f"Epoch del mejor mIoU: {history['val_miou'].index(max(history['val_miou'])) + 1}")

## 8. Cargar Mejor Checkpoint

In [ ]:
from pathlib import Path

# Cargar mejor checkpoint
checkpoint_path = Path(f'../models/checkpoints/{EXPERIMENT_NAME}/best_checkpoint.pth')
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Mejor checkpoint cargado (epoch {checkpoint['epoch']})")
print(f"  mIoU: {checkpoint['best_val_miou']:.4f}")

## 9. Visualizar Predicciones

In [ ]:
import torch.nn.functional as F

# Poner modelo en eval
model.eval()

# Tomar un batch de validación
images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

# Predecir
with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

# Visualizar primeros 4 samples
n_samples = 5
fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))

for i in range(n_samples):
    # Imagen
    img_np = images[i].cpu().permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Imagen')
    axes[i, 0].axis('off')
    
    # Ground Truth
    axes[i, 1].imshow(masks[i].cpu().numpy(), cmap='tab10', vmin=0, vmax=8)
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')
    
    # Predicción
    axes[i, 2].imshow(preds[i].cpu().numpy(), cmap='tab10', vmin=0, vmax=8)
    axes[i, 2].set_title('Predicción')
    axes[i, 2].axis('off')
    
    # Overlay
    axes[i, 3].imshow(img_np)
    axes[i, 3].imshow(preds[i].cpu().numpy(), cmap='tab10', alpha=0.4, vmin=0, vmax=8)
    axes[i, 3].set_title('Overlay')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 10. Métricas Detalladas en Validación

In [ ]:
from scripts.utils import SegmentationMetrics, CLASS_NAMES

# Crear métrica
metrics = SegmentationMetrics(num_classes=N_CLASSES, ignore_index=0, class_names=CLASS_NAMES)

# Evaluar en todo el validation set
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        metrics.update(preds, masks)

# Imprimir resumen
metrics.print_summary()

## ✅ Conclusiones

- Entrenamiento completado exitosamente
- Checkpoint guardado en `models/checkpoints/`
- Métricas calculadas por clase

**Próximos pasos:**
1. Evaluar en test set (notebook 03)
2. Entrenar otros modelos (DeepLabv3+, SegFormer)
3. Comparar resultados
4. Post-procesamiento y vectorización